# Job Task 3: Pipeline Completion Summary

Prints a final row-count summary confirming the pipeline completed successfully.
This output is visible in the job run log and in the email notification.

In [ ]:
dbutils.widgets.text("catalog",       "platform-workshop", "Catalog")
dbutils.widgets.text("shared_schema", "00_shared",         "Shared Schema")

catalog       = dbutils.widgets.get("catalog")
shared_schema = dbutils.widgets.get("shared_schema")
c = f"`{catalog}`"

In [ ]:
from datetime import datetime

bronze_count  = spark.sql(f"SELECT COUNT(*) AS cnt FROM {c}.{shared_schema}.financial_docs_bronze").collect()[0]['cnt']
silver_count  = spark.sql(f"SELECT COUNT(*) AS cnt FROM {c}.{shared_schema}.financial_docs_silver").collect()[0]['cnt']
gold_count    = spark.sql(f"SELECT COUNT(*) AS cnt FROM {c}.{shared_schema}.financial_docs_gold").collect()[0]['cnt']
summary_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {c}.{shared_schema}.company_ai_investment_summary").collect()[0]['cnt']

print("=" * 50)
print("  Financial Intelligence Pipeline — Complete")
print(f"  Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M UTC')}")
print("=" * 50)
print(f"  Bronze  (raw PDFs):            {bronze_count:>6,} rows")
print(f"  Silver  (parsed + extracted):  {silver_count:>6,} rows")
print(f"  Gold    (classified):          {gold_count:>6,} rows")
print(f"  Summary (aggregated insights): {summary_count:>6,} rows")
print("=" * 50)

# Company breakdown from gold
print("\n  Documents by company:")
company_counts = spark.sql(
    f"SELECT company, COUNT(*) AS docs FROM {c}.{shared_schema}.financial_docs_gold "
    "WHERE company IS NOT NULL GROUP BY company ORDER BY company"
).collect()
for row in company_counts:
    print(f"    {row['company']:<20} {row['docs']:>4} documents")